<a href="https://colab.research.google.com/github/vishaljoshi24/DungeonsAndDragonsTurnClassification/blob/Codex-Branch/prompt_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone -b Codex-Branch https://github.com/vishaljoshi24/DungeonsAndDragonsTurnClassification/
%cd /content/DungeonsAndDragonsTurnClassification

In [ ]:
!pip install dspy==3.2.1

In [ ]:
import pandas as pd
import dspy

In [ ]:
training_df = pd.read_excel('further_testing_articulation_codes_set.xlsx')

In [ ]:
training_df

In [ ]:
training_df.drop(columns=['Jack\'s Codes'], inplace=True)

In [ ]:
training_df.drop(columns=['Agreed Codes'], inplace=True)

In [ ]:
training_df.drop(columns=['Notes'], inplace=True)

In [ ]:
training_df

In [ ]:
trainset = []

for context, current_turn, category in training_df.values:
    examples = dspy.Example(context=context, question=current_turn, response=category).with_inputs("context", "question")
    trainset.append(examples)

In [ ]:
trainset

In [ ]:
lm = dspy.LM('ollama_chat/qwen3:8b', api_base = 'http://localhost:11434', api_key='', max_tokens=2048)
dspy.configure(lm=lm)

In [ ]:
from typing import Literal


class TurnClassifier(dspy.Signature):
    """Given the context for a Dungeons & Dragons game turn and the game turn itself, classify the turn."""
    context: str = dspy.InputField(desc = "The three previous game turns which describe a player's action or their dialogue.")
    question: str = dspy.InputField (desc="The current turn taken by a player, which can include a description of an action or a piece of dialogue.")
    response: Literal['knowledge request',
                      'knowledge update',
                      'knowledge share',
                      'knowledge confirmation',
                      'argumentation',
                      'resource use',
                      'resource share',
                      'resource aid',
                      'resource request',
                      'enact narration',
                      'deterministic action',
                      ] = dspy.OutputField()

class PlayerInstruction(dspy.Signature):
  context: str = dspy.InputField(desc = "The three previous game turns which describe a player's action or their dialogue.")
  question: str = dspy.InputField (desc="The current turn taken by a player, which can include a description of an action or a piece of dialogue.")
  category: Literal['knowledge request',
                      'knowledge update',
                      'knowledge share',
                      'knowledge confirmation',
                      'argumentation',
                      'resource use',
                      'resource share',
                      'resource aid',
                      'resource request',
                      'enact narration',
                      'deterministic action',
                    ] = dspy.InputField()
  player_instruction:str = dspy.OutputField(desc="instruction on how to behave within a D&D game.")

In [ ]:
class ClassifyTurns(dspy.Module):
  def __init__(self):
    self.classifier = dspy.ChainOfThought(TurnClassifier, caching=False)

  def forward(self, context, question, **kwargs):
    prediction = self.classifier(context=context, question=question)
    return prediction


In [ ]:
classify = ClassifyTurns()
classify.load("optimized_classifier(1).json")

In [ ]:
class PromptGenerator(dspy.Module):
  def __init__(self):
    # self.classifier = classify
    self.generator = dspy.ChainOfThought(PlayerInstruction, caching=False)

  def forward(self, context, question, category, **kwargs):
    # pred_category = classify(context=context, question=question).response
    prompt = self.generator(context=context, question=question, category=category)
    return prompt, category

prompt_generator = PromptGenerator()


In [ ]:
prompt_generator(trainset[1]['context'], trainset[1]['question'], trainset[1]['response'])

In [ ]:
new_df = training_df.drop_duplicates('Vishal\'s Codes')

In [ ]:
new_df

In [ ]:
new_set = []

for context, current_turn, category in new_df.values:
    examples = dspy.Example(context=context, question=current_turn, response=category).with_inputs("context", "question")
    new_set.append(examples)

In [ ]:
new_set

In [ ]:
generated_prompt = []

for i in range(len(trainset)):
    generated_prompt.append(prompt_generator(trainset[i]['context'], trainset[i]['question'], trainset[i]['response']))

In [ ]:
generated_prompt

In [ ]:
generated_prompt[1][0]['player_instruction']

In [ ]:
category = []
player_instruction = []

for i in range(len(generated_prompt)):
  category.append(generated_prompt[i][1])
  player_instruction.append(generated_prompt[i][0]['player_instruction'])


In [ ]:
category

In [ ]:
player_instruction

In [ ]:
instruction_library = list(zip(category, player_instruction))

In [ ]:
instruction_library

In [ ]:
instruction_df = pd.DataFrame(instruction_library)

In [ ]:
instruction_df.drop_duplicates(subset=[0], inplace=False)

In [ ]:
filename = 'instructions.xlsx'
instruction_df.to_excel(filename)